# 02: Etap 1, MotionNet (globalny ruch kamery)

**Status: wytrenowane 2026-10-13** (stary notatnik `archiwum/train_colab_2026-10-13.ipynb`, Colab T4, 30 epok).
Wagi: `MyDrive/rs_project/runs/2026-10-13_motion_dual/best.pt` (kopia z dawnego `MyDrive/rs_runs/`).

| run | PSNR | SSIM | EPE |
|---|---|---|---|
| bez korekcji | 19.66 | 0.527 | – |
| motion_dual | **28.51** | **0.874** | **0.57 px** |
| motion_single | 19.90 | 0.544 | 3.98 px |
| flow_dual | 20.45 | 0.588 | 3.15 px |

Ten notatnik służy do ponownej ewaluacji i ewentualnego treningu od nowa (np. na zbiorze z parametrami zmierzonymi na kamerach).

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'LOGIN/NAZWA_REPO'      # <- UZUPEŁNIJ RAZ: repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
P.migrate_old_layout()     # jednorazowo kopiuje stare MyDrive/rs_runs/* -> rs_project/runs/2026-10-13_*
P.status()

In [ ]:
# Zbiór danych: rozpakowanie z Drive na dysk maszyny Colaba (szybki odczyt w trakcie treningu)
if not os.path.exists('/content/dataset/train'):
    z = P.find_dataset_zip()
    assert z, 'Brak zbioru na Drive - najpierw uruchom 01_dataset.ipynb'
    print('rozpakowuję', z, '...')
    !unzip -q -n "{z}" -d /content
print(len(os.listdir('/content/dataset/train')), 'próbek train /', len(os.listdir('/content/dataset/val')), 'val')
WORKERS = min(8, os.cpu_count())

## Trening (uruchamiaj tylko, gdy trenujesz od nowa)
Nazwa runu zawiera datę, więc nowy trening nie nadpisze starego. Po rozłączeniu: dodaj `--resume <folder>/last.pt`.

In [ ]:
from datetime import date
RUN = f'{date.today()}_motion_dual'
!python src/train.py --data /content/dataset --mode motion --epochs 30 --bs 16 \
    --workers {WORKERS} --out "{P.RUNS}" --name {RUN}

In [ ]:
# ablacje (do pracy): tylko kamera A, oraz U-Net bez geometrii
!python src/train.py --data /content/dataset --mode motion --single --epochs 30 --bs 16 --workers {WORKERS} --out "{P.RUNS}" --name {date.today()}_motion_single
!python src/train.py --data /content/dataset --mode flow --epochs 30 --bs 16 --workers {WORKERS} --out "{P.RUNS}" --name {date.today()}_flow_dual

## Ewaluacja wszystkich runów etapu 1 (zapisuje `eval.json` w folderze runu)

In [ ]:
import json, pandas as pd
rows = []
for name in ['motion_dual', 'motion_single', 'flow_dual']:
    ck = P.find_run(name)
    if ck is None:
        print('brak', name); continue
    out = ck.parent / 'eval.json'
    !python src/train.py --data /content/dataset --eval "{ck}" --eval-json "{out}" --workers {WORKERS} > /dev/null
    m = json.load(open(out))['metrics']
    rows.append({'run': ck.parent.name, **{k: round(v, 3) for k, v in m.items()}})
pd.DataFrame(rows).set_index('run')